# Track #2 — RT-DETR-l  ·  Kaggle (T4 ×2)

**Setup (right sidebar) before running:**
1. **Accelerator** → `GPU T4 x2`  ·  **Internet** → `On`
2. **Add Input → Competitions** → add this competition (data auto-found under `/kaggle/input/`). *No upload needed.*
3. In **cell 1**, set `REPO_URL` to your GitHub repo. *(Repo name / folder layout don't matter — scripts are auto-located.)*

**How to run:** Run cells **1→4** (setup + smoke test). Paste me the smoke-test output to confirm.
Then run the **one** 🚀 heavy cell (cell 5) and go to sleep — it trains all 3 folds **and** writes
`submission_rtdetr.csv`, refreshing it **after every fold** (so a valid CSV always exists, even if the
session stops early). Cell 6 is only needed if you want to (re)generate the CSV separately.


## 1 · Clone repo + install  *(edit REPO_URL)*


In [ ]:
REPO_URL = 'https://github.com/Shimul-Baidya/duetAIHackathon_my_sub.git'   # already set to your repo

import os, sys, glob, shutil, subprocess
REPO_DIR = '/kaggle/working/' + REPO_URL.rstrip('/').split('/')[-1].replace('.git','')
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)          # always pull the latest code
subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_DIR], check=True)
# auto-locate scripts dir (works whatever the repo is named / structured)
hits = glob.glob(f'{REPO_DIR}/**/rtdetr_lib.py', recursive=True)
assert hits, f'rtdetr_lib.py not found under {REPO_DIR} -- is REPO_URL correct?'
SCRIPTS = os.path.dirname(hits[0])
if SCRIPTS not in sys.path: sys.path.insert(0, SCRIPTS)
for m in ('rtdetr_lib','submission_utils','ensemble_tracks'):  # drop any stale cached import
    sys.modules.pop(m, None)
print('scripts:', SCRIPTS)

subprocess.run(['pip','-q','install','ultralytics','albumentations'], check=True)
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


## 2 · Config + build dataset


In [ ]:
import rtdetr_lib as L

SRC      = L.find_competition_src()      # auto-finds /kaggle/input/<comp> (has train/train.csv)
DS_OUT   = '/kaggle/working/yolo_ds'
RUNS     = '/kaggle/working/runs'
OUT      = '/kaggle/working'
TEST_DIR = f'{SRC}/test/images'

MODEL    = 'rtdetr-l.pt'   # COCO-pretrained, auto-downloads (Internet must be ON)
K        = 3
IMGSZ    = 1280           # used by inference here; training imgsz is pinned in _train_fold.py
BATCH    = 8              # FYI only — training batch/imgsz are pinned in _train_fold.py to the
                          #   config proven on fold 0 (imgsz 1280 / batch 8 -> ~10.9 GB/GPU).
                          #   To change them, edit _train_fold.py (single source of truth).
DEVICE   = '0,1'          # BOTH GPUs (DDP). If the SMOKE ever fails, set DEVICE='0' (single GPU,
                          #   no DDP) — then the launcher can't hit the DDP re-init crash.
EPOCHS   = 100
PATIENCE = 20
FOLDS    = [0, 1, 2]      # each trains in its own process (see HEAVY cell)
print('SRC =', SRC)

DS = L.build_dataset(SRC, DS_OUT, k=K)


## 3 · 🔬 SMOKE TEST (~10-12 min) — proves the run will NOT stop after fold 0
The previous run died after fold 0 because Ultralytics **cannot re-initialise DDP a 2nd time in the
same Python process**. The fix: train each fold in its **own** process (`_train_fold.py` via
`subprocess`) — fresh interpreter + fresh DDP every time, so the crash is structurally impossible.

This smoke trains folds **0, 1, 2** for 2 epochs each (the SAME launcher the heavy cell uses, just
fewer epochs) and validates inference. It deliberately launches folds 1 & 2 — the ones that died.
**It must print `✅ ALL PASS`.** It also confirms IMGSZ=1280 / BATCH=8 fits memory.


In [ ]:
# 🔬 SMOKE — proves the multi-fold launcher (the exact thing that died after fold 0).
# Trains folds 0,1,2 for 2 epochs EACH in SEPARATE processes (fresh DDP every time), then
# checks the inference path. ~10-12 min. Must end with "✅ ALL PASS".
import subprocess, sys, os
res = []
for k in [0, 1, 2]:
    print(f'\n--- SMOKE fold {k} (fresh DDP process) ---')
    rc = subprocess.run([sys.executable, f'{SCRIPTS}/_train_fold.py', str(k), DS, RUNS, 'smoke'],
                        cwd=SCRIPTS).returncode
    ok = (rc == 0 and os.path.exists(f'{RUNS}/smoke_f{k}/weights/best.pt'))
    res.append(ok)
    print(f'   fold {k}: rc={rc}  best.pt={ok}  ->  {"PASS" if ok else "FAIL"}')
assert all(res), '❌ multi-fold launch STILL broken — do NOT run the heavy cell; switch DEVICE to "0"'
c = L.predict_one(f'{RUNS}/smoke_f0/weights/best.pt', TEST_DIR, imgsz=IMGSZ, device='0')
print(f'\n✅ ALL PASS — 3 folds trained in separate processes; inference OK ({len(c)} test imgs).')
print('   Crash-after-fold-0 cannot recur. Run the HEAVY cell next.')


## 4 · 🚀 HEAVY CELL — run once, then sleep
Trains every fold in `FOLDS` (epochs=100), **each in its own process** (same `_train_fold.py`
launcher the smoke proved). After **each** fold it (re)writes `/kaggle/working/submission_rtdetr.csv`
from all folds done so far (WBF-fused) — so a valid CSV always exists even if the session stops early.
A failed fold is skipped, not fatal. ~6-7 h for 3 folds (fits the 12 h limit).


In [ ]:
# 🚀 HEAVY — folds in FOLDS (epochs=100), each in its OWN process (same launcher as smoke).
# Submission is refreshed after every fold, so a valid CSV always exists even if cut short.
import subprocess, sys, os
done = []
for k in FOLDS:
    print(f'\n=============== TRAIN FOLD {k} (epochs={EPOCHS}) ===============')
    rc = subprocess.run([sys.executable, f'{SCRIPTS}/_train_fold.py', str(k), DS, RUNS, 'full'],
                        cwd=SCRIPTS).returncode
    if rc == 0 and os.path.exists(f'{RUNS}/rtdetr_l_f{k}/weights/best.pt'):
        done.append(k)
        try:
            sub = L.infer_all_folds(RUNS, TEST_DIR, OUT, imgsz=IMGSZ, device='0')
            L.validate_submission(sub, TEST_DIR)
            print(f'>>> submission refreshed using folds {done} -> {sub}')
        except Exception as e:
            print(f'[WARN] inference after fold {k} failed: {e!r} — will retry next fold')
    else:
        print(f'[WARN] fold {k} failed (rc={rc}) — continuing to next fold')
print(f'\nDONE. folds trained: {done}. Submission: {OUT}/submission_rtdetr.csv')


## 5 · (only if needed) regenerate the CSV from saved weights
Run this on wake-up if the heavy cell got interrupted *after* training but before writing the CSV.
Uses whatever `best.pt` files exist — no retraining.


In [ ]:
sub = L.infer_all_folds(RUNS, TEST_DIR, OUT, imgsz=IMGSZ, device='0')
L.validate_submission(sub, TEST_DIR)
import pandas as pd; print(); print(pd.read_csv(sub).head(3).to_string())


## 6 · Submit
Download **`/kaggle/working/submission_rtdetr.csv`** and submit it. ✅ Complete Track-2 entry.

Per-fold prediction caches are in `/kaggle/working/caches/` for the optional grand ensemble with
Track #1 later (`track2/scripts/ensemble_tracks.py` — accepts Track #1's `submission.csv` directly).
